# 06. Hands-on: Amazon Bedrock Intelligent Prompt Routing

The labs so far (the SGLang decision layer) take the approach of **"removing the decision LLM call entirely"**. Bedrock **Intelligent Prompt Routing (IPR)** is the managed feature that instead **"chooses which model each generation call goes to"**.

| Comparison | Decision layer (labs 01-05) | Intelligent Prompt Routing |
|---|---|---|
| What it does | Answers with one label instantly (prefill-only) | Predicts request difficulty and picks the small/large model within a family |
| Target | The decision nodes inside the agent | The LLM call that generates the customer reply |
| Effect | Decision latency in tens of ms, decision cost ~0 | Sends easy requests to the small model, cutting generation cost |
| Relationship | **Complements** — remove decisions with the decision layer, then route the remaining generation calls with IPR | |

Calling it is simple: put the **router ARN** in the `modelId` of `converse()`, and read which model was selected from `trace.promptRouter.invokedModelId` in the response.


## 1. Prompt routers in this region

The region provides **default routers** per model family, and you can create **custom routers** with `CreatePromptRouter`.

> **Model note (as of 2026-10)**: the default Anthropic router references Claude 3 Haiku / 3.5 Sonnet, which are **EOL** (calls are rejected), and the current Claude 4.5 family is **not yet supported** by prompt routing (CreatePromptRouter returns a ValidationException — verified directly). This notebook therefore runs the live lab on the **Nova family (Lite/Pro)**. The mechanism and the API are identical regardless of family.


In [ ]:
import boto3, json, os, time, statistics
from IPython.display import Markdown, display

# region/account resolved dynamically (workshop notebook instance = us-east-1)
REGION = os.environ.get("BEDROCK_REGION") or boto3.session.Session().region_name or "us-east-1"
ACCT = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
br = boto3.client("bedrock", region_name=REGION)
rt = boto3.client("bedrock-runtime", region_name=REGION)
print(f"region={REGION} account={ACCT}")

rows = ["| Router | type | qualityDiff | member models |", "|---|---|---|---|"]
for r in br.list_prompt_routers(type="custom")["promptRouterSummaries"]:
    models = ", ".join(m["modelArn"].split("/")[-1] for m in r["models"])
    rows.append(f"| {r['promptRouterName']} | {r['type']} | "
                f"{r['routingCriteria']['responseQualityDifference']} | {models} |")
display(Markdown("\n".join(rows)))

## 2. Send the same support questions through the router

We send three English support questions of increasing difficulty to the **Nova default router** and observe **which model gets selected** per request, plus the latency. Observe which model each request gets routed to. Routing is prompt-dependent: with these English phrasings the router's picks can differ from the Korean run (in our executed copy the short greeting went to Pro while the longer questions went to Lite) — treat the quality-difference knob in the next section as the lever you actually control.


In [ ]:
DEFAULT_NOVA = f"arn:aws:bedrock:{REGION}:{ACCT}:default-prompt-router/amazon.nova:1"

QUESTIONS = [
    ("greeting/FAQ (easy)", "Hello, what are your business hours?"),
    ("card inquiry (medium)", "My card doesn't swipe. What should I do? Keep it brief."),
    ("remittance inquiry (hard)", "My overseas remittance hasn't arrived for two days. "
                        "Lay out the verification steps for SWIFT intermediary fee deductions, "
                        "beneficiary-bank cutoffs, and AML holds, plus what to tell the customer."),
]

def route_once(router_arn, text, max_tokens=200):
    t0 = time.perf_counter()
    r = rt.converse(modelId=router_arn,
                    messages=[{"role": "user", "content": [{"text": text}]}],
                    inferenceConfig={"maxTokens": max_tokens})
    ms = (time.perf_counter() - t0) * 1000
    picked = r.get("trace", {}).get("promptRouter", {}).get("invokedModelId", "?").split("/")[-1]
    answer = r["output"]["message"]["content"][0]["text"]
    return picked, ms, answer, r["usage"]

for tag, q in QUESTIONS:
    print(f"[{tag}]")
    print(f"  customer question: {q}")
    picked, ms, answer, usage = route_once(DEFAULT_NOVA, q)
    print(f"  selected model: {picked}   ({ms:,.0f}ms, in {usage['inputTokens']}/out {usage['outputTokens']} tok)")
    print(f"  answer: {answer.strip()[:160]}")
    print()


## 3. Creating a custom router — tuning the routing appetite

`routingCriteria.responseQualityDifference` (%, multiples of 5) is the tolerance: "use the small model even if its predicted quality is this much lower than the large one". Larger values route more aggressively to the small model.

The custom router below (tolerance 25%) already exists in this account. Its creation code, for reference (no need to re-run):

```python
br.create_prompt_router(
    promptRouterName="ws-nova-router",
    routingCriteria={"responseQualityDifference": 25},   # %, multiples of 5 only
    models=[{"modelArn": ".../inference-profile/us.amazon.nova-lite-v1:0"},
            {"modelArn": ".../inference-profile/us.amazon.nova-pro-v1:0"}],
    fallbackModel={"modelArn": ".../inference-profile/us.amazon.nova-pro-v1:0"})
```


In [ ]:
# custom router: create in this account if missing (get-or-create)
CUSTOM_NAME = "ws-nova-router"
found = [r for r in br.list_prompt_routers(type="custom")["promptRouterSummaries"]
         if r["promptRouterName"] == CUSTOM_NAME]
if found:
    CUSTOM_NOVA = found[0]["promptRouterArn"]
else:
    CUSTOM_NOVA = br.create_prompt_router(
        promptRouterName=CUSTOM_NAME,
        routingCriteria={"responseQualityDifference": 25},
        models=[{"modelArn": f"arn:aws:bedrock:{REGION}:{ACCT}:inference-profile/us.amazon.nova-lite-v1:0"},
                {"modelArn": f"arn:aws:bedrock:{REGION}:{ACCT}:inference-profile/us.amazon.nova-pro-v1:0"}],
        fallbackModel={"modelArn": f"arn:aws:bedrock:{REGION}:{ACCT}:inference-profile/us.amazon.nova-pro-v1:0"},
    )["promptRouterArn"]
print("custom router:", CUSTOM_NOVA)


rows = ["| question | default (diff=0) | custom (diff=25) |", "|---|---|---|"]
for tag, q in QUESTIONS:
    p1, m1, _, _ = route_once(DEFAULT_NOVA, q)
    p2, m2, _, _ = route_once(CUSTOM_NOVA, q)
    rows.append(f"| {tag} | {p1.replace('us.amazon.','')} ({m1:,.0f}ms) "
                f"| {p2.replace('us.amazon.','')} ({m2:,.0f}ms) |")
display(Markdown("\n".join(rows)))

## 4. How to use it together with the decision layer

- **Decisions (D1-D8)**: IPR does not solve these — whichever model it routes to, one decision is still one LLM generation call (hundreds of ms, billed). Decisions belong to the decision layer (tens of ms, ~zero cost).
- **Generation (respond, summarize)**: this is IPR's home turf. Let the small model write the easy replies and cut generation cost.
- Example combination: `guardrail/routing/escalation = SGLang decisions` + `final reply generation = IPR (auto small/large selection)` — cutting decision latency and generation cost at the same time.

**Caveats**
- The router's difficulty prediction is itself imperfect; for quality-sensitive work start with a conservative responseQualityDifference (0-10).
- IPR routes **within one family only** (no Claude<->Nova mixing).
- Latency is dominated by the selected model's generation time — IPR is a cost optimizer, not a latency remover like the decision layer.
- Pricing: IPR charges **$1 / 1,000 requests** on top of the selected model's token cost (aws.amazon.com/bedrock/pricing). That is a fixed $0.001 per decision-like call, so for high-volume "decision" use the cost structure is fundamentally different from a decision layer (~zero marginal cost).
